# 01 · 候选池构建

**目标**：对 `etf_universe.json` 的 28 只初筛 ETF 做程序化校核，输出通过预过滤的最终候选池。

**筛选规则**（详见 `基金组合优化参考.md` §一.1）：
1. 上市满 5 年（`start_date <= 2021-09-08`）
2. 截至当前仍处于上市状态
3. 近 250 个交易日日均成交额 > 5000 万元
4. 两两相关系数 > 0.92 时保留日均成交更高者

**输入**：`etf_portfolio/etf_universe.json`

**输出**：`etf_portfolio/outputs/universe.csv`

In [ ]:
# ============================================================
# cell 0: imports + 全局参数
# ============================================================
from jqdata import *            # 聚宽 magic：get_all_securities / get_price / get_security_info
import sys, os, json
from pathlib import Path
from datetime import datetime, date

import pandas as pd
import numpy as np

# 项目根目录
PROJ = Path(os.environ.get('PROJ_ROOT', Path.cwd())).resolve()
if not (PROJ / 'etf_portfolio').exists():
    # 兜底：用 notebook 所在目录的父目录
    PROJ = Path('/Users/huhao/src/codesnip/python/ai/028-jukuan').resolve()
sys.path.insert(0, str(PROJ))

from etf_portfolio.universe import load_initial_universe, filter_candidates

# 全局参数
AS_OF              = date(2026, 9, 8)         # 筛选基准日
MIN_YEARS          = 5.0                      # 最小上市年限
MIN_AVG_MONEY      = 5.0e7                    # 最小日均成交额 (元)
CORR_THRESH        = 0.92                     # 相关去重阈值
TURNOVER_LOOKBACK  = 250                      # 成交额回看天数
CORR_LOOKBACK      = 750                      # 相关矩阵回看天数

OUTPUT_DIR = PROJ / 'etf_portfolio' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_PATH  = PROJ / 'etf_portfolio' / 'etf_universe.json'

print(f'PROJ      = {PROJ}')
print(f'OUTPUT_DIR= {OUTPUT_DIR}')
print(f'JSON_PATH = {JSON_PATH}')

In [ ]:
# ============================================================
# cell 1: 加载初筛池 + 运行预过滤
# ============================================================
init_uni = load_initial_universe(JSON_PATH)
print(f'初筛池共 {len(init_uni)} 只 ETF')
init_uni[:3]

In [ ]:
# ============================================================
# cell 2: 程序化校核（5 年上市 + 5000 万成交 + 0.92 相关去重）
# ============================================================
result = filter_candidates(
    init_uni,
    as_of=AS_OF,
    min_years=MIN_YEARS,
    min_avg_money=MIN_AVG_MONEY,
    corr_thresh=CORR_THRESH,
    lookback_days=TURNOVER_LOOKBACK,
    corr_lookback_days=CORR_LOOKBACK,
    verbose=True,
)

In [ ]:
# ============================================================
# cell 3: 查看剔除明细
# ============================================================
dropped_df = pd.DataFrame(
    [(code, reason) for code, reason in result.dropped.items()],
    columns=['code', 'reason']
).sort_values('code').reset_index(drop=True)
print(f'共剔除 {len(dropped_df)} 只')
dropped_df

In [ ]:
# ============================================================
# cell 4: 最终入池 ETF
# ============================================================
init_lookup = {x['code']: x for x in init_uni}
kept_df = pd.DataFrame([
    {
        'code':      code,
        'name':      init_lookup[code]['name'],
        'category':  init_lookup[code]['category'],
        'list_date': init_lookup[code]['list_date'],
    }
    for code in result.kept if code in init_lookup
])
print(f'最终入池 {len(kept_df)} 只')
kept_df

In [ ]:
# ============================================================
# cell 5: 持久化
# ============================================================
out_csv = OUTPUT_DIR / 'universe.csv'
kept_df.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'已写入 {out_csv}')

# 相关矩阵（带 name 索引）
if result.corr_matrix is not None and not result.corr_matrix.empty:
    corr_out = OUTPUT_DIR / 'corr_matrix.csv'
    corr_named = result.corr_matrix.rename(columns=init_lookup, index=init_lookup)
    corr_named.to_csv(corr_out, encoding='utf-8-sig')
    print(f'已写入 {corr_out}')

In [ ]:
# ============================================================
# cell 6: 相关矩阵热图（可选）
# ============================================================
import matplotlib.pyplot as plt

if result.corr_matrix is not None and not result.corr_matrix.empty:
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(result.corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(result.corr_matrix.columns)))
    ax.set_xticklabels([init_lookup.get(c, c) for c in result.corr_matrix.columns], rotation=90)
    ax.set_yticks(range(len(result.corr_matrix.index)))
    ax.set_yticklabels([init_lookup.get(c, c) for c in result.corr_matrix.index])
    plt.colorbar(im, ax=ax, label='相关系数')
    ax.set_title('候选 ETF 日收益相关系数矩阵')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'corr_heatmap.png', dpi=120)
    plt.show()

## 中间结论

- 通过预过滤的 ETF 数量应为 15–25 只
- 被剔除的常见原因：上市不满 5 年 / 日均成交额不足 / 与其他 ETF 相关度过高
- 进入下一阶段（`02_数据准备与收益矩阵.ipynb`）需引用 `universe.csv`